# NPC patient 1 — finished ablation weights

Slide `20210809-C-AH4199551` only. These are the 2026-11-17 weights in `result_uni2_ablation_<arm>_new1066axisfix_paired_v2_seed42`, not the older A2 / A7 / A8 trees.

Predict first, on a GPU that is not training:

```bash
CUDA_DEVICE=2 bash code/External_val/NPC/demo_uni2_axisfix.sh predict
```

The question for the main text is whether an expert TLS is called B cell. On the three leftmost foci (26 Visium spots, 5,262 nuclei), cell2location puts B first: mean proportion 0.29 inside versus 0.13 outside, and B wins 19 of the 26 spots. None of these weights do that, including `l12_links_2` and `l12_balanced` (argmax B inside the outline is 0). For the first seven, argmax of `B_cell` or `Plasma` inside the outline is 0 to 0.08%. Mean `P(B_cell)+P(Plasma)` is lower inside the outline than outside (ratio 0.42–0.63). The spatial arms call the foci DC: `spatial_k8_gate` 79% DC, `l12_links_1` 66% DC, `spatial_k8` 48% DC. The `l2_only` decision rule does not move them to B (DC bias is positive, B_cell bias is negative). These seven maps are not the main-text TLS panel.

`l12_links_2`, `l12_balanced`, and `focal_gamma1` are in this notebook. `gate_l12_balanced` is not: that run is still training.

The no-neighbor arms (`legacy_nospatial`, `adjacent_nospatial`, `l12_balanced_nospatial`) are not shown.

In [ ]:
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
for _p in (REPO / "code", REPO / "code" / "Hist2Pheno_pkg", REPO / "code" / "External_val" / "NPC"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from Hist2Pheno_pkg.panel_figures import crop_he_rectangle
from npc_figures import draw_tls, keep_leftmost_foci, load_tls_xy, nuclei_in_contours, tls_smooth_contours

SAMPLE = "20210809-C-AH4199551"
NPC = REPO / "data" / "Visium" / "NPC"
# Same left-hand TLS foci as Pred_visual_npc.ipynb.
TLS_PATH = NPC / "Annotation" / "TLS_spot_label.csv"
VISIUM_DIR = NPC / "patient1"
HE = NPC / "HE_images" / f"{SAMPLE}.tif"
B_NAMES = ("B_cell", "Plasma")

# Neighbor arms only. legacy_nospatial, adjacent_nospatial, and l12_balanced_nospatial are left out.
ARMS = [
    ("spatial_k8", "trunk: adjacent + k=8 mean"),
    ("spatial_k8_gate", "trunk + spatial gate"),
    ("l12_links_1", "trunk, L12 link weights = 1"),
    ("l12_links_2", "trunk, L12 link weights = 2"),
    ("l12_balanced", "trunk + balanced L12"),
    ("focal_gamma1", "trunk + focal γ=1, class weights kept"),
    ("balanced_sampler", "trunk + sampler, no class weight"),
]

def pred_dir(arm):
    return NPC / f"pred_uni2_axisfix_{arm}" / SAMPLE

## L2 prediction on the whole slide

One panel per finished arm. Points are the L2 argmax (`pred_CT`) on the patient1 H&E. The color scale is shared. The TLS crop is below; this figure is the full section.

In [ ]:
from matplotlib.lines import Line2D

from Hist2Pheno_pkg.external_validation import simple_cohort

cohort = simple_cohort("NPC", REPO, embedder="uni2")
thumb, full_w, full_h = cohort._he_thumbnail(HE, max_side=1100)
palette = cohort._palette()["L2"]
dpi = 110
point_s = (0.9 * 72.0 / dpi) ** 2

panels = []
for arm, note in ARMS:
    path = pred_dir(arm) / f"{SAMPLE}_stardist_pred.h5ad"
    if not path.is_file():
        panels.append((arm, note, None))
        continue
    panels.append((arm, note, ad.read_h5ad(path)))

ncols = 3
nrows = (len(panels) + 1 + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15.2, 5.15 * nrows), dpi=dpi)
axes = np.atleast_1d(axes).ravel()
present = set()
for ax, (arm, note, adata) in zip(axes, panels):
    if adata is None:
        ax.set_title(f"{arm}\nmissing")
        ax.set_axis_off()
        continue
    labels = adata.obs["pred_CT"].astype(str).to_numpy()
    xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
    present.update(map(str, labels.tolist()))
    cohort._scatter_on_he(
        ax, thumb, full_w, full_h, xy, labels, palette,
        f"{arm}\n{note}  n={adata.n_obs:,}",
        point_s,
        show_he=True,
    )
for ax in axes[len(panels):]:
    ax.set_axis_off()
order = [name for name in palette if name in present]
order += sorted(present - set(order))
axes[len(panels)].legend(
    [
        Line2D(
            [0], [0], linestyle="none", marker="o", markersize=7,
            markerfacecolor=palette.get(name, (0.5, 0.5, 0.5, 1))[:3],
            markeredgecolor="none", label=name,
        )
        for name in order
    ],
    order, loc="center", frameon=False, fontsize=8,
)
fig.suptitle(f"{SAMPLE}  L2 cell type", fontsize=13)
fig.tight_layout()
out = NPC / "pred_uni2_axisfix_figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / f"{SAMPLE}_L2_axisfix.png", dpi=140, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
tls_xy, tls_bc, pitch_px, spot_r_px = load_tls_xy(TLS_PATH, VISIUM_DIR)
keep = keep_leftmost_foci(tls_xy, pitch_px, n_keep=3)
tls_xy, tls_bc = tls_xy[keep], tls_bc[keep]
contours = tls_smooth_contours(tls_xy, disk_r=0.70 * pitch_px, sigma=0.45 * pitch_px)
pad = 2.6 * pitch_px
x0 = max(0.0, float(tls_xy[:, 0].min() - pad))
y0 = max(0.0, float(tls_xy[:, 1].min() - pad))
Image.MAX_IMAGE_PIXELS = None
with Image.open(HE) as im:
    full_w, full_h = im.size
x1 = min(float(full_w), float(tls_xy[:, 0].max() + pad))
y1 = min(float(full_h), float(tls_xy[:, 1].max() + pad))
he_roi = crop_he_rectangle(HE, x0, y0, x1, y1)
print(f"TLS foci {len(contours)}, zoom {x1 - x0:.0f} x {y1 - y0:.0f} px, HE {he_roi.shape}")


In [ ]:
rows = []
loaded = {}
for arm, note in ARMS:
    folder = pred_dir(arm)
    h5 = folder / f"{SAMPLE}_stardist_pred.h5ad"
    npz = folder / f"{SAMPLE}_l2_probs.npz"
    if not h5.is_file() or not npz.is_file():
        rows.append({"arm": arm, "note": note, "status": "missing"})
        continue
    adata = ad.read_h5ad(h5)
    pack = np.load(npz, allow_pickle=True)
    names = [str(x) for x in pack["class_names"]]
    probs = np.asarray(pack["probs"], dtype=np.float32)
    if probs.shape[0] != adata.n_obs:
        raise RuntimeError(f"{arm}: probs {probs.shape[0]} != h5ad {adata.n_obs}")
    b_idx = [names.index(n) for n in B_NAMES]
    p_b = probs[:, b_idx].sum(axis=1)
    pred = adata.obs["pred_CT"].astype(str).to_numpy()
    xy = np.asarray(adata.obsm["spatial_HE"], dtype=np.float64)
    inside = nuclei_in_contours(xy, contours)
    called = np.isin(pred, list(B_NAMES))
    rows.append({
        "arm": arm,
        "note": note,
        "n_inside": int(inside.sum()),
        "B softmax inside": float(p_b[inside].mean()),
        "B softmax outside": float(p_b[~inside].mean()),
        "B argmax inside": float(called[inside].mean()),
        "B argmax outside": float(called[~inside].mean()),
    })
    loaded[arm] = {"p_b": p_b, "xy": xy, "inside": inside, "pred": pred}

table = pd.DataFrame(rows)
if "B softmax inside" in table.columns:
    table["inside / outside"] = table["B softmax inside"] / table["B softmax outside"]
display_cols = [c for c in table.columns if c != "note"]
print(table[display_cols].to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
for _, row in table.iterrows():
    print(f"{row['arm']}: {row['note']}")


## B-cell probability inside the expert TLS

Each panel is `P(B_cell) + P(Plasma)` on the same TLS crop. The color scale is shared, so a brighter focus is more B mass, not a different colormap. The black line is the expert outline. A panel that is bright inside the line and quiet outside it is the one to carry into the main text. Argmax labels are in the table; this figure does not hide B behind a Tumor winner.

In [ ]:
ready = [arm for arm, _ in ARMS if arm in loaded]
if not ready:
    raise FileNotFoundError(
        "No patient-1 predictions yet. Run: "
        "CUDA_DEVICE=2 bash code/External_val/NPC/demo_uni2_axisfix.sh predict"
    )
vmax = float(np.quantile(np.concatenate([loaded[a]["p_b"] for a in ready]), 0.99))
ncols = 4
nrows = (len(ARMS) + 1 + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16.5, 4.1 * nrows), dpi=120)
axes = axes.ravel()
last = None
for ax, (arm, note) in zip(axes, ARMS):
    ax.imshow(he_roi, extent=(0, he_roi.shape[1], he_roi.shape[0], 0), interpolation="bilinear")
    if arm not in loaded:
        ax.set_title(f"{arm}\nmissing")
        ax.set_axis_off()
        continue
    pack = loaded[arm]
    xy = pack["xy"]
    in_zoom = (
        (xy[:, 0] >= x0) & (xy[:, 0] < x1) & (xy[:, 1] >= y0) & (xy[:, 1] < y1)
    )
    sc = ax.scatter(
        xy[in_zoom, 0] - x0,
        xy[in_zoom, 1] - y0,
        c=pack["p_b"][in_zoom],
        s=2.0,
        cmap="magma",
        vmin=0.0,
        vmax=vmax,
        linewidths=0,
        rasterized=True,
    )
    last = sc
    draw_tls(ax, contours, local=(x0, y0), fill=False, lw=1.2)
    ax.set_title(arm, fontsize=10)
    ax.set_xlim(0, he_roi.shape[1])
    ax.set_ylim(he_roi.shape[0], 0)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes[len(ARMS):]:
    ax.set_axis_off()
if last is not None:
    fig.colorbar(last, ax=axes[len(ARMS)], fraction=0.6, label="P(B_cell)+P(Plasma)")
fig.suptitle("patient1 expert TLS  —  B-cell probability, shared scale", fontsize=12)
fig.tight_layout()
out = NPC / "pred_uni2_axisfix_figures"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / f"{SAMPLE}_TLS_B_seven.png", dpi=160)
plt.show()